In [56]:
import Implementation_OFLP as IO
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import math
from pyscipopt import Model, quicksum
scip = Model()

In [55]:
def points_to_cons(points : list, p_inside : list) -> tuple[list, list]:
    A = []
    b = []
    n = len(p)
    # We want constriants of the form Ax <= b, where x = [x,y]
    for i in range(p):
        x1,y1,x2,y2 = points[i][0], points[i][1], points[(i + 1) % n][0], points[(i + 1) % n][1]
        new_A = [y2 - y1 , x1 - x2]
        new_b = x1 * y2 - x2 * y1
        if new_A @ p_inside > new_b:
            A.append(- new_A)
            b.append(- new_b)
        else:
            A.append(new_A)
            b.append(new_b)
    return A, b

def feas_region_constraint(x_lb, x_ub, y_lb, y_ub, circles, method : int, model, var) -> None:
    '''
    This function creates the constraint for the feasible region.
    :param method: This is either 0 or 1 and it denotes the methods, 0 is LP, and 1 is GS.
    '''
    if method == 0:
        solutions = IO.generate_points(x_lb, x_ub, y_lb, y_ub)
        A,b = IO.feas_region_box_LP(x_lb, x_ub, y_lb, y_ub, circles, solutions)
    sorted_points, deleted_points = IO.feas_region_box_GS(x_lb, x_ub, y_lb, y_ub, circles)
    if len(deleted_points) == 0:
        n = len(sorted_points)
        p1, p2 = sorted_points[0], sorted_points[math.ceil(n/2)]
        x1, x2 = p1[0], p1[1]
        y1, y2 = p2[0], p2[1]
        deleted_points = [[x1 + ((x2 - x1) / 2), y1 + ((y2 - y1) / 2)]]
    A, b = points_to_cons(sorted_points, deleted_points[0])
    for i in range(len(b)):
        model.addCons(A[i] * var <= b[i], f'Convex hull constraint {i} for {var}')
    return None

def distance(p1, p2) -> float:
    return (p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2


def create_objective(model, var_x, var_y, circles):
    L = model.addVar(vtype='C', lb=0, ub=None, name='L')
    for j, c in enumerate(circles):
        for i, (x, y) in enumerate(zip(var_x, var_y)):
            model.addCons(distance(c,(x,y)) >= L, f"MaxDistance_{i}_With{c[0],c[1]}")
    model.setObjective(L,'maximize')
    return None
    

def create_LP(x_lb : float, x_ub : float, y_lb : float, y_ub : float, circles : list, facilities : int, min_fac_dist : float, mode : int) -> None:

    OFP = Model(f'OFP problem with {facilities} facilities (with minimal distance {min_fac_dist}) and {len(circles)} communities')
    ObnFacilities_x = [OFP.addVar(vtype='C', lb=x_lb, ub=x_ub, name=f"x_{i}") for i in range(facilities)]
    ObnFacilities_y = [OFP.addVar(vtype='C', lb=y_lb, ub=y_ub, name=f"y_{i}") for i in range(facilities)]
    for i in range(facilities):
        var = [ObnFacilities_x[i], ObnFacilities_y[i]]
        feas_region_constraint(x_lb, x_ub, y_lb, y_ub, circles, mode, OFP, var)
        for j in range(i, facilities):
            var2 = [ObnFacilities_x[j], ObnFacilities_y[j]]
            scip.addCons(distance(var, var2) >= min_fac_dist, f'MinDistF{i}_F{j}')
    create_objective(model, ObnFacilities_x,ObnFacilities_y, circles)
    return OFP

def generate_a_solution(facilities, communities) -> float:
    return None



import re
from pathlib import Path

try:
    BASE_DIR = Path(__file__).parent
except NameError:
    BASE_DIR = Path.cwd()
INSTANCE_DIR = BASE_DIR / "initialized_instances"

SET_RE    = re.compile(r"(\w+)\s+\w+\s*/\s*(\d+)\*(\d+)\s*/")
PARAM_RE  = re.compile(r"community_([xy])\('(\d+)'\)\s*=\s*([-+0-9.eE]+)\s*;")
SCALAR_RE = re.compile(r"Parameter\s+(\w+)\s*/\s*([-+0-9.eE]+)\s*/")
BOUND_RE  = re.compile(r"\b(x|y|facility_facility_sq_distance)\.(lo|up)\([^)]*\)\s*=\s*([^;]+);")

def read_instance(filename: str) -> dict:
    '''Reads a GAMS instance: facilities, communities, box bounds and facility-facility bounds.'''
    text = (INSTANCE_DIR / filename).read_text()

    sets = {name: int(end) - int(start) + 1 for name, start, end in SET_RE.findall(text)}

    coords = {"x": {}, "y": {}}
    for axis, idx, value in PARAM_RE.findall(text):
        coords[axis][int(idx)] = float(value)
    communities = [(coords["x"][k], coords["y"][k]) for k in range(1, sets["j"] + 1)]

    scalars = {name: float(value) for name, value in SCALAR_RE.findall(text)}
    bounds = {}
    for var, side, value in BOUND_RE.findall(text):
        value = value.strip()
        bounds.setdefault(var, {})[side] = scalars.get(value) if value in scalars else float(value)

    return {
        "facilities": sets["i"],
        "x_bounds": (bounds["x"]["lo"], bounds["x"]["up"]),
        "y_bounds": (bounds["y"]["lo"], bounds["y"]["up"]),
        "ff_sq_bounds": (bounds["facility_facility_sq_distance"]["lo"],
                         bounds["facility_facility_sq_distance"]["up"]),
        "communities": communities
    }

def natural_key(path):
    '''Sorts "instance2" before "instance10" by comparing numbers as numbers.'''
    return [int(part) if part.isdigit() else part for part in re.split(r"(\d+)", path.name)]

instances = sorted(
    (f for f in INSTANCE_DIR.iterdir() if f.is_file()),
    key=natural_key,
)
    


def run(instance : dict, mode: int) -> None:
    '''
    This functions runs our program
    '''  
    facilities = instance['facilities']
    x_lb, x_ub = instance['x_bounds']
    y_lb, y_ub = instance['y_bounds']
    min_FF_dist = instance['ff_sq_bounds'][0]
    communities = instance['communities']
    #Init_sol = generate_a_solution(facilities, communities, x_lb, x_ub, y_lb, y_ub, min_FF_dist, communities)
    Init_sol = 0.01
    circles = []
    for i in communities:
        circles.append([i[0], i[1], Init_sol])
    print(circles)
    OFP = create_LP(x_lb, x_ub, y_lb, y_ub, circles, facilities, min_FF_dist, mode)
    OFP.optimize()

inst_loaded = read_instance(instances[0])
run(inst_loaded, 1)

[[0.00097, 0.00367, 0.01], [0.85243, 0.84373, 0.01], [0.84217, 0.53687, 0.01], [0.47523, 0.01453, 0.01], [0.83537, 0.54207, 0.01], [0.38603, 0.55333, 0.01], [0.90057, 0.13927, 0.01], [0.06483, 0.74013, 0.01], [0.15777, 0.64847, 0.01], [0.79163, 0.65493, 0.01], [0.92697, 0.58967, 0.01], [0.64643, 0.17773, 0.01], [0.72817, 0.68287, 0.01], [0.50923, 0.98853, 0.01], [0.28137, 0.84807, 0.01], [0.06003, 0.56733, 0.01], [0.50657, 0.20527, 0.01], [0.77883, 0.19413, 0.01], [0.52377, 0.07447, 0.01], [0.94563, 0.94893, 0.01], [0.65297, 0.97567, 0.01], [0.64043, 0.71173, 0.01], [0.41417, 0.62887, 0.01], [0.74323, 0.16253, 0.01], [0.52737, 0.95407, 0.01], [0.93403, 0.78133, 0.01], [0.91257, 0.07127, 0.01], [0.69283, 0.84813, 0.01], [0.68977, 0.30047, 0.01], [0.29963, 0.44293, 0.01], [0.17897, 0.16167, 0.01], [0.83443, 0.44573, 0.01], [0.90017, 0.37487, 0.01], [0.17723, 0.53653, 0.01], [0.57337, 0.86007, 0.01], [0.00803, 0.19533, 0.01], [0.11857, 0.73727, 0.01], [0.80683, 0.70213, 0.01], [0.65577, 0

TypeError: 'int' object is not subscriptable

In [43]:
F2C100 = read_instance(instances[0])
F2C100






{'facilities': 2,
 'x_bounds': (0.0, 1.0),
 'y_bounds': (0.0, 1.0),
 'ff_sq_bounds': (0.25, 2.0),
 'communities': [(0.00097, 0.00367),
  (0.85243, 0.84373),
  (0.84217, 0.53687),
  (0.47523, 0.01453),
  (0.83537, 0.54207),
  (0.38603, 0.55333),
  (0.90057, 0.13927),
  (0.06483, 0.74013),
  (0.15777, 0.64847),
  (0.79163, 0.65493),
  (0.92697, 0.58967),
  (0.64643, 0.17773),
  (0.72817, 0.68287),
  (0.50923, 0.98853),
  (0.28137, 0.84807),
  (0.06003, 0.56733),
  (0.50657, 0.20527),
  (0.77883, 0.19413),
  (0.52377, 0.07447),
  (0.94563, 0.94893),
  (0.65297, 0.97567),
  (0.64043, 0.71173),
  (0.41417, 0.62887),
  (0.74323, 0.16253),
  (0.52737, 0.95407),
  (0.93403, 0.78133),
  (0.91257, 0.07127),
  (0.69283, 0.84813),
  (0.68977, 0.30047),
  (0.29963, 0.44293),
  (0.17897, 0.16167),
  (0.83443, 0.44573),
  (0.90017, 0.37487),
  (0.17723, 0.53653),
  (0.57337, 0.86007),
  (0.00803, 0.19533),
  (0.11857, 0.73727),
  (0.80683, 0.70213),
  (0.65577, 0.32647),
  (0.85363, 0.13693),
  (0.50

In [30]:
model = Model()
n = 10
x = [
    model.addVar(vtype='C', lb=0, name=f'facility{i}')
    for i in range(n)]

cons_1 = [model.addCons(x[i] + x[(i+1)%10] <= 3, name=f'Cons_{i}') for i in range(n)
        ]
cons_2 = [model.addCons(x[i] - x[(i + 3) %n] == 0) for i in range(3,n)]
    
model.setObjective(np.sum(x), sense='maximize')
model.optimize()

feasible solution found by trivial heuristic after 0.0 seconds, objective value 0.000000e+00
presolving:
(round 1, fast)       7 del vars, 7 del conss, 0 add conss, 10 chg bounds, 0 chg sides, 0 chg coeffs, 0 upgd conss, 0 impls, 0 clqs, 0 implints
(round 2, fast)       7 del vars, 8 del conss, 0 add conss, 11 chg bounds, 0 chg sides, 0 chg coeffs, 0 upgd conss, 0 impls, 0 clqs, 0 implints
(round 3, exhaustive) 7 del vars, 14 del conss, 0 add conss, 11 chg bounds, 0 chg sides, 0 chg coeffs, 0 upgd conss, 0 impls, 0 clqs, 0 implints
   (0.0s) symmetry computation started: requiring (bin +, int +, cont +), (fixed: bin -, int -, cont -)
   (0.0s) symmetry computation finished: 1 generators found (max: 1500, log10 of symmetry group size: 0.30) (symcode time: 0.00)
dynamic symmetry handling statistics:
   orbitopal reduction:       no components
   orbital reduction:         no components
   lexicographic reduction:   no permutations
handled 1 out of 1 symmetry components
(round 4, exhausti

# Running the main program

In [ ]:
def create_LP(sorted_points : list)

4
5
